In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5  # Low-level frames
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import linregress

## Functions

In [ ]:
def FD_moments(data,peakthresh,maxFD,RM_arr,M0=True,M1=True,M2=False,*args,**kwargs):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use = data[(abs(RM_arr) <= maxFD),:,:]    
    RM_arr_use = RM_arr[(abs(RM_arr) <= maxFD)]  

    # set any data points below PI threshold to NaN:
    data_use[data_use < peakthresh] = np.nan

    moments = {}
    if M0:
        M0_data = dFD*np.nansum(data_use,axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        M1_data = dFD*np.nansum(data_use*RM_arr_use[:,np.newaxis,np.newaxis],axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use[:,np.newaxis,np.newaxis]-M1_data)**2,axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

def clean_up_header(hdu):

    hdr_new = fits.Header()
    for card in hdu[0].header.cards:
        if card.keyword.strip() != "":
            hdr_new.append(card)

    return hdr_new

In [ ]:
def read_files_4channels(directory,stokes,filetype):

    band = ['A','B','C','D']
    data_list = []
    hdr_list = []
    print('Reading in '+stokes+' for filetype: '+filetype)

    for i in range(0,4):
        print('band '+band[i])
        hdu = fits.open(directory+stokes+band[i]+'_'+filetype+'.fits')
        data_list.append(hdu[0].data)

        hdr = fits.Header()
        for card in hdu[0].header.cards:
            if card.keyword.strip() != "":
                hdr.append(card)
        hdr['OBJECT'] = stokes+band[i]+'_'+filetype
        hdr_list.append(hdr)
        #print(repr(hdr))
        #print('-------------------')

    gc.collect()

    return data_list,hdr_list

In [ ]:
def calc_pol_angles(Q_list,U_list,hdrQ_list,hdrU_list,keep_PA=True,
                    savefile=False,*args,**kwargs):
    
    PA = []
    zz = []
    dtau = []
    descrptn = []
    hdr_PA = []
    
    for i in range(0,len(Q_list)):
        
        descrptn.append('PA_'+hdrQ_list[i]['OBJECT'][1:])
        
        PA.append(0.5*np.arctan2(U_list[i],Q_list[i]))
        
        ## correct angle wrap:
        zz.append(np.exp(2*PA[i]*1j))
        if i > 0:
            dtau.append(np.arctan2((zz[i]*np.conj(zz[i-1])).imag,
                                   (zz[i]*np.conj(zz[i-1])).real)/2.)
            PA[i] = PA[i-1]+dtau[i-1]
            
        hdr_new = hdrQ_list[i].copy()
        hdr_new['OBJECT'] = descrptn[i]
        hdr_PA.append(hdr_new)
        
        print('finished calculating PA for array '+str(i+1)+' of '+str(len(Q_list)))
            
    if savefile:
        for i in range(0,len(PA)):
            fits.writeto(directory+descrptn[i]+'.fits',PA[i],
                         header=hdr_PA[i],overwrite=True)
    
    del zz; del dtau
    if keep_PA == False:
        del PA
        PA = 0
    gc.collect()
    
    return PA, hdr_PA

In [ ]:
def calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=False,*args,**kwargs):
    
    freq = np.array([1406.9,1413.8,1427.4,1434.3])
    lbd2 = ((3e8)/(freq*1e6))**2
    
    RM     = np.empty_like(PA_list[0])
    PAint  = np.empty_like(PA_list[0])
    rvalue = np.empty_like(PA_list[0])
    pvalue = np.empty_like(PA_list[0])
    stderr = np.empty_like(PA_list[0])
    intstd = np.empty_like(PA_list[0])
    
    descrptn_RM = 'RM_'+hdrPA_list[0]['OBJECT'][5:]
    
    for i in range(0,RM.shape[0]):
        print(str(i)+' of '+str(RM.shape[0])+' lat pixels')
        for j in range(0,RM.shape[1]):
            if np.isfinite(PA_list[0][i,j]):
                PA = np.array([PA_list[0][i,j],PA_list[1][i,j],
                               PA_list[2][i,j],PA_list[3][i,j]])
                #print(PA)
                try:
                    result = linregress(lbd2,PA)
                    #print(result.slope)
                    RM[i,j]     = result.slope
                    PAint[i,j]  = result.intercept
                    rvalue[i,j] = result.rvalue
                    pvalue[i,j] = result.pvalue
                    stderr[i,j] = result.stderr
                    intstd[i,j] = result.intercept_stderr
                except:
                    pass
                del PA
        print('------------------------------------------')

        gc.collect()
            
    hdr_RM = hdrPA_list[0].copy()
    hdr_RM['OBJECT'] = descrptn_RM
    hdu_list = fits.HDUList()
    
    hdr_RM['EXTNAME'] = 'RM'
    hdu1 = fits.ImageHDU(RM, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'PA_INT'
    hdu2 = fits.ImageHDU(PAint, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'RVALUE'
    hdu3 = fits.ImageHDU(rvalue, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'PVALUE'
    hdu4 = fits.ImageHDU(pvalue, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'STDERR'
    hdu5 = fits.ImageHDU(stderr, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'INT_STDERR'
    hdu6 = fits.ImageHDU(intstd, header=hdr_RM)

    for hdu_app in [hdu1,hdu2,hdu3,hdu4,hdu5,hdu6]:
        print(hdu_app)
        hdu_list.append(hdu_app)

    if savefile:        
        hdu_list.writeto(directory+descrptn_RM+'.fits',overwrite=True)
    
    if keep_RM == False:
        del RM; del hdu_list; del PAint
        del rvalue; del pvalue; del stderr; del intstd
        RM = 0; hdu_list = 0
    gc.collect()
    
    return RM, hdu_list

## Read in GMIMS linear fit RMs and clean up header

In [ ]:
use_regridded = True

if use_regridded:
    #hdu_G_RM = fits.open('/srv/aordog/cgps_gmims_data/RM_G_conv4_regrd.fits')
    hdu_G_RM = fits.open('/srv/data/cgps-gmims/conv_regrid/RM_G_regrd.fits')
    G_RM = hdu_G_RM[0].data
    print(G_RM.shape)

    hdr_G_RM = clean_up_header(hdu_G_RM)
    del hdr_G_RM['CROTA1']
    hdr_G_RM['WCSAXES'] = 2
    print(repr(hdr_G_RM))

    wcs = WCS(hdr_G_RM)

    y_coords, x_coords = np.indices((hdr_G_RM['NAXIS2'], hdr_G_RM['NAXIS1']))
    G_RM_coords = wcs.all_pix2world(x_coords, y_coords, 0)
    G_RM_l = G_RM_coords[0][0]
    G_RM_b = G_RM_coords[1][:,0]

    del G_RM_coords; del y_coords; del x_coords
    gc.collect()

## Read in GMIMS FD cube, clean up header and make 2D version

In [ ]:
hdu_G_FD = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_FD_PI.fits')
G_FD = hdu_G_FD[0].data
print(G_FD.shape)

hdr_G_FD = clean_up_header(hdu_G_FD)
del hdr_G_FD['WCSAXES']
del hdr_G_FD['LONPOLE']
del hdr_G_FD['LATPOLE']
del hdr_G_FD['ORIGIN']
del hdr_G_FD['INSTRUME']
del hdr_G_FD['COMMENT']
del hdr_G_FD['HISTORY']
print(repr(hdr_G_FD))

hdr_G_FD_2D = hdr_G_FD.copy()
hdr_G_FD_2D['NAXIS'] = 2
del hdr_G_FD_2D['NAXIS3']
del hdr_G_FD_2D['CRPIX3']
del hdr_G_FD_2D['CDELT3']
del hdr_G_FD_2D['CUNIT3']
del hdr_G_FD_2D['CTYPE3']
del hdr_G_FD_2D['CRVAL3']
print('---------------------')
print(repr(hdr_G_FD_2D))

hdr_G_FD_2Dfix = hdr_G_FD_2D.copy()
hdr_G_FD_2Dfix['CRPIX1'] = 1
hdr_G_FD_2Dfix['CRPIX2'] = 1
hdr_G_FD_2Dfix['CRVAL1'] = 360
hdr_G_FD_2Dfix['CRVAL2'] = -90
#hdr_G_FD_2Dfix[''] = 
#hdr_G_FD_2Dfix[''] =
print('---------------------')
print(repr(hdr_G_FD_2Dfix))

## Calculate GMIMS full moments and peaks

In [ ]:
FD_idx = np.linspace(0,hdr_G_FD['NAXIS3']-1,hdr_G_FD['NAXIS3'])
FD_ax = hdr_G_FD['CRVAL3']+(FD_idx - hdr_G_FD['CRPIX3']+1)*hdr_G_FD['CDELT3']
print(FD_ax.shape)

G_moments = FD_moments(G_FD,0.03,400,FD_ax)
G_M1 = G_moments['M1']

peak_FD = np.empty_like(G_FD[0,:,:])
print(peak_FD.shape)
for i in range(0,peak_FD.shape[0]):
    for j in range(0,peak_FD.shape[1]):
        idx = np.where(G_FD[:,i,j] == np.nanmax(G_FD[:,i,j]))[0][0]
        peak_FD[i,j] = FD_ax[idx]

## Calculate GMIMS linear fits

In [ ]:
#data_listQ, hdr_listQ = read_files_4channels('/srv/aordog/cgps_gmims_data/','Q','GMIMS_W21_HR')
#data_listU, hdr_listU = read_files_4channels('/srv/aordog/cgps_gmims_data/','U','GMIMS_W21_HR')

data_listQ, hdr_listQ = read_files_4channels('/srv/data/cgps-gmims/gmims_OG/','Q','GMIMS_W21_HR')
data_listU, hdr_listU = read_files_4channels('/srv/data/cgps-gmims/gmims_OG/','U','GMIMS_W21_HR')

PA_list, hdrPA_list = calc_pol_angles(data_listQ,data_listU,
                                      hdr_listQ,hdr_listU,
                                      keep_PA=True,savefile=False)
        
RM,hdu_list = calc_RM(PA_list,hdrPA_list,keep_RM=True,savefile=False)


In [ ]:
#hdu_G_fitpeak = fits.open('/srv/aordog/cgps_gmims_data/fitted_peaks_GphiPeakPIfit_rm2.fits')
hdu_G_fitpeak = fits.open('/srv/data/cgps-gmims/gmims_FD/fitted_peaks_GphiPeakPIfit_rm2.fits')
G_fitpeak = hdu_G_fitpeak[0].data
print(G_fitpeak.shape)

In [ ]:
plt.imshow(RM,origin='lower',vmin=-500,vmax=500,cmap='RdBu_r')

In [ ]:
fig = plt.figure(figsize=(20,10))

plt.imshow(G_fitpeak,origin='lower',vmin=-50,vmax=50,cmap='RdBu_r')

In [ ]:
fig = plt.figure(figsize=(20,10))

plt.imshow(RM/G_fitpeak,origin='lower',vmin=-10,vmax=10,cmap='RdBu_r')

In [ ]:
fig = plt.figure(figsize=(16,6))

ax1 = fig.add_subplot(211)
ax1.hist(RM[G_moments['M0']>50].flatten(),bins=501,range=(-300,300),alpha=0.7,color='C1');
ax1.set_xlim(-200,200)
#ax1.set_xlabel(r'RM and $\sigma_{RM}$')
ax1.grid()

ax2 = fig.add_subplot(212)
ax2.hist(G_fitpeak[G_moments['M0']>50].flatten(),bins=501,range=(-300,300),color='C0');
ax2.set_xlim(-200,200)
#ax1.set_xlabel(r'RM and $\sigma_{RM}$')
ax2.grid()


In [ ]:
fig = plt.figure(figsize=(12,6))

ax1 = fig.add_subplot(121)
ax1.hist2d(RM[G_moments['M0']>50].flatten(),G_fitpeak[G_moments['M0']>50].flatten(),
           range=([[-500,500],[-100,100]]),bins=(201,201), cmap='cubehelix',vmin=0,vmax=100);
ax1.set_xlim(-500,500)
ax1.set_ylim(-100,100)
#ax1.set_xlabel('RM')
#ax1.set_ylabel(r'$\sigma_{RM}$')

ax2 = fig.add_subplot(122)
ax2.scatter(RM.flatten(),G_fitpeak.flatten(),s=1);
ax2.set_xlim(-700,700)
ax2.set_ylim(-200,200)

## Check maps

In [ ]:
fig = plt.figure(figsize=(20, 9))

wcs = WCS(hdr_G_RM)
print(wcs)
c = SkyCoord([192,52], [-5,5], frame=Galactic, unit="deg")

ax = fig.add_axes(311, projection=wcs)
ax.imshow(G_RM, vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
ax.set_xlim(wcs.world_to_pixel(c)[0])
ax.set_ylim(wcs.world_to_pixel(c)[1])

plt.show()

In [ ]:
peak_FD_reproj, footprint = reproject_interp((peak_FD,WCS(hdr_G_FD_2D)), WCS(hdr_G_FD_2Dfix))



fig = plt.figure(figsize=(20, 9))

wcs = WCS(hdr_G_FD_2Dfix)
print(wcs)
c = SkyCoord([192,60], [-5,5], frame=Galactic, unit="deg")

ax = fig.add_axes(311, projection=wcs.celestial)
ax.imshow(peak_FD_reproj, vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
ax.set_xlim(wcs.world_to_pixel(c)[0])
ax.set_ylim(wcs.world_to_pixel(c)[1])

plt.show()

In [ ]:
fig = plt.figure(figsize=(20, 9))

c = SkyCoord([192,50], [-5,7], frame=Galactic, unit="deg")

lon_use = c.l.wrap_at(180*u.deg).deg
lat_use = c.b.deg

c2 = SkyCoord(lon_use, lat_use, frame=Galactic, unit="deg")

wcs = WCS(hdr_G_RM)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0]) ; i2 = int(x[1]) ; j1 = int(y[0]) ; j2 = int(y[1])
plt.subplot(311,projection=wcs[j1:j2,i1:i2])
plt.imshow(G_RM[j1:j2,i1:i2], vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
#plt.xlabel('Galactic Longitude') ; plt.ylabel('Galactic Latitude')

wcs = WCS(hdr_G_FD_2D)
print(wcs)
x, y = np.round(wcs.world_to_pixel(c2))
i1 = int(x[0]) ; i2 = int(x[1]) ; j1 = int(y[0]) ; j2 = int(y[1])
print(i1,i2,j1,j2)
plt.subplot(312,projection=wcs[j1:j2,i1:i2])
plt.imshow(peak_FD[j1:j2,i1:i2], vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
#plt.xlabel('Galactic Longitude') ; plt.ylabel('Galactic Latitude')


In [ ]:
fs = 18
fig = plt.figure(figsize=(20, 5))

c = SkyCoord([192,50], [-5,7], frame=Galactic, unit="deg")

wcs = WCS(hdu_G_RM[0].header)
x, y = np.round(wcs.world_to_pixel(c))
i1 = int(x[0])
i2 = int(x[1])
j1 = int(y[0])
j2 = int(y[1])

plt.subplot(211,projection=wcs[j1:j2,i1:i2])
plt.imshow(G_RM[j1:j2,i1:i2], vmin=-100, vmax=100, origin='lower',cmap='RdBu_r')
#plt.contour(G_RM[j1:j2,i1:i2], levels=[PI_lim], colors='white', alpha=0.5)
plt.xlabel('Galactic Longitude')
plt.ylabel('Galactic Latitude')

#plt.subplot(212,projection=wcs[j1:j2,i1:i2])
#plt.imshow(peak_FD_reproj[j1:j2,i1:i2], vmin=-50, vmax=50, origin='lower',cmap='RdBu_r')
#plt.xlabel('Galactic Longitude')
#plt.ylabel('Galactic Latitude')

In [ ]:
print(FD_ax[60:141])
print(FD_ax[140])

In [ ]:
FD_interp = np.empty([len(FD_ax[60:141]),peak_FD_reproj.shape[0],peak_FD_reproj.shape[1]])
print(FD_interp.shape)

for i in range(60,141):
    print(i,FD_ax[i])
    FD_interp[i,:,:], footprint = reproject_interp((G_FD[i,:,:],hdrFD_2D), hdu_G_RM[0].header)